# DOCX → Azota (Colab T4) — chạy từ đầu

**Notebook cũ `Untitled1.ipynb` sai:** clone 2 lần, thiếu ImageMagick, `pip --only-binary` fail trên Python 3.13.

1. Runtime → Disconnect and delete runtime
2. Upload **file này** hoặc mở từ repo `colab_start_here.ipynb`
3. Runtime → T4 GPU
4. `Shift+Enter` từng ô. **Không** Run all.

Cấm: `pip install unimernet[full]` / `tokenizers` / `transformers==4.42.4`.


## 1. GPU


In [ ]:
!nvidia-smi -L


## 2. Clone code (một lần)


In [ ]:
import shutil, sys
from pathlib import Path
for p in ("/content/repo", "/content/docx-to-azota", "/content/refurbished-marketplace"):
    shutil.rmtree(p, ignore_errors=True)
REPO = "https://github.com/phuchoang2603/refurbished-marketplace.git"
BRANCH = "cursor/docx-to-azota-pipeline-4d56"
!git clone -b {BRANCH} --depth 1 {REPO} /content/repo
shutil.copytree("/content/repo/tools/docx-to-azota", "/content/docx-to-azota")
sys.path.insert(0, "/content/docx-to-azota")
print("OK", Path("/content/docx-to-azota/convert.py").exists())


## 3. Import converter (chưa cần UniMERNet)


In [ ]:
from convert import convert_docx, apply_unimernet_latex
from eval_timer import StepTimer
from colab_opt import detect_profile, prepare_unimernet_checkpoint, free_cuda
from colab_opt import vision_jobs_from_manifest, inject_latex_into_markup
from vision import rasterize_formula_image, load_unimernet, unimernet_batch

NAME, PROFILE = detect_profile()
timer = StepTimer()
OUT = "/content/azota_out"
print(NAME, PROFILE)


## 4. ImageMagick (WMF → PNG) — đợi xong apt


In [ ]:
!pip -q install pillow pymupdf huggingface_hub
!apt-get -qq install -y imagemagick libmagickwand-dev
!pip -q install Wand
print("ImageMagick OK")


## 5. UniMERNet — `--no-deps`, không `[full]`, không `--only-binary`

`install_unimernet_colab()` vá `transformers.onnx` trên đĩa rồi mới import.


In [ ]:
from install_colab import allow_wmf_in_imagemagick, install_unimernet_colab
allow_wmf_in_imagemagick()
install_unimernet_colab()


## 6. Upload đề `.docx`


In [ ]:
from google.colab import files
uploaded = files.upload()
DOCX = "/content/" + next(iter(uploaded))
print(DOCX)


## 7. Bước 1 — extract Azota (CPU, bắt buộc). Có thể dừng sau ô này.


In [ ]:
from pathlib import Path
with timer.step("Bước 1", "OOXML"):
    man = convert_docx(DOCX, OUT)
print(man["counts"])
print("\n".join(Path(OUT, "markup.txt").read_text(encoding="utf-8").splitlines()[:30]))


## 8. Bước 2 — WMF MathType → PNG (tùy chọn, cần `$latex$`)


In [ ]:
from pathlib import Path
png_dir = Path(OUT) / "sidecar_png"
png_dir.mkdir(exist_ok=True)
jobs = []
with timer.step("Bước 2", "raster WMF"):
    for aid, src in vision_jobs_from_manifest(man, OUT, kinds=("mathtype",)):
        dest = png_dir / f"{aid}.png"
        got = rasterize_formula_image(src, dest, dpi=200)
        if got:
            jobs.append((aid, got))
print(len(jobs), "ảnh công thức")


## 9. Tải UniMERNet-tiny (ô lâu)


In [ ]:
with timer.step("Bước 3-load", "download tiny"):
    cfg = prepare_unimernet_checkpoint("tiny", "/content/models")
    model, vis, device = load_unimernet(cfg_path=cfg, fp16=True)
print("device =", device)


## 10. Nhận dạng LaTeX + gắn vào markup


In [ ]:
from pathlib import Path
with timer.step("Bước 3", "UniMERNet batch"):
    preds = unimernet_batch(model, vis, device, jobs, batch_size=8)
apply_unimernet_latex(man, preds, Path(OUT))
p = Path(OUT) / "markup.txt"
text = p.read_text(encoding="utf-8")
with timer.step("Bước 4", "inject LaTeX"):
    text = inject_latex_into_markup(text, preds)
    p.write_text(text, encoding="utf-8")
timer.print_summary()
for k, v in list(preds.items())[:5]:
    print(k, "→", v[:100])
free_cuda(model, vis)
print("đã unload UniMERNet")


## 11. Tải zip (bỏ OCR hình trên T4)


In [ ]:
from google.colab import files
!cd /content && zip -qr azota_out.zip azota_out
files.download("/content/azota_out.zip")
